In [17]:
import numpy as np
import plotly.graph_objects as go

In [18]:
L = 1.0 # length [m]
t_final = 1.0 # final time [s]
alpha = 0.1 # diffusivity [m^2/s]
POSITIONS = (0.1* L, 0.25*L, 0.5*L, 0.65*L, 0.85*L)

## 3.1 

In [19]:
def temperature(x, t):
    """Closed form solution for the one-dimensional heat equation."""
    return np.sin(np.pi * np.asarray(x) / L) * np.exp(
        -alpha * (np.pi / L) ** 2 * np.asarray(t)
    )

In [20]:
times = np.linspace(0.0, t_final, 201)
positions = np.linspace(0.0, L, 201)

trajectories = go.Figure()
styles = ("solid", "solid", "solid", "dash", "dash")
for x, style in zip(POSITIONS, styles):
    trajectories.add_trace(
        go.Scatter(
            x=times,
            y=temperature(x, times),
            mode="lines",
            name=f"x = {x:g}",
            line={"dash": style},
        )
    )
trajectories.update_layout(
    title="Temperature at five fixed positions",
    xaxis={"title": "Time t", "range": [0, t_final]},
    yaxis={"title": "Temperature u(x, t)"},
    legend_title="Rod position",
    )

In [21]:
values = temperature(positions[:, None], times[None, :])
heatmap = go.Figure(
    data=go.Heatmap(
        x=times,
        y=positions,
        z=values,
        zmin=0,
        zmax=1,
        colorscale="Inferno",
        colorbar={"title": "Temperature u(x, t)"},
    )
)
heatmap.update_layout(
    title="Temperature in the rod",
    xaxis={"title": "Time t"},
    yaxis={"title": "Position x"},
)

## Task 3.3

In [22]:
import torch
import torch.nn as nn


class Net(nn.Module):
    def __init__(self):
        super(Net,self).__init__()
        activation = nn.Tanh()
        self.net = nn.Sequential(
            nn.Linear(2, 10),
            activation,
            nn.Linear(10, 20),
            activation,
            nn.Linear(20, 10),
            activation,
            nn.Linear(10, 1),
        )

    def forward(self, tensor):
        return self.net(tensor)


In [23]:
def generate_training_points(n, L, T):
    x_interior = torch.rand(n, 1)*L
    x_interior.requires_grad_(True)
    t_interior = torch.rand(n, 1)*T
    t_interior.requires_grad_(True)

    t_boundary = torch.rand(n, 1)*T
    x_left = torch.zeros_like(t_boundary)
    x_right = torch.full_like(t_boundary, L)

    target_boundary = torch.zeros_like(t_boundary)

    x_initial = torch.rand(n,1)*L
    t_initial = torch.zeros_like(x_initial)

    target_initial = torch.sin(torch.pi * x_initial / L)
    
    return {"x_interior": x_interior, "t_interior": t_interior, "x_left": x_left, "x_right": x_right, "t_boundary": t_boundary, "x_initial": x_initial, "t_initial": t_initial, "target_boundary": target_boundary, "target_initial": target_initial}

In [24]:
def residual(model, x,t,alpha):
    u = model(torch.cat([x, t], dim=1))  
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]

    return u_t - alpha *u_xx


In [25]:
def compute_losses(model, pts, alpha):
    mse = nn.MSELoss()
    res = residual(model, pts["x_interior"], pts["t_interior"], alpha)
    loss_res = mse(res, torch.zeros_like(res))

    u_left = model(torch.cat([pts["x_left"], pts["t_boundary"]], dim=1))
    u_right = model(torch.cat([pts["x_right"], pts["t_boundary"]], dim=1))

    loss_boundary = mse(u_left, pts["target_boundary"]) + mse(u_right, pts["target_boundary"])

    u_initial =  model(torch.cat([pts["x_initial"], pts["t_initial"]], dim=1))
    loss_initial = mse(u_initial, pts["target_initial"])

    return loss_res, loss_boundary, loss_initial

In [26]:
# L2 error vs closed form

xs = torch.linspace(0, L, 101)
ts = torch.linspace(0, t_final, 101)
X, Tt = torch.meshgrid(xs, ts, indexing="ij")
grid = torch.stack([X.flatten(), Tt.flatten()], dim=1)

def temperature_torch(x, t):
    return torch.sin(torch.pi * x / L) * torch.exp(-alpha * (torch.pi / L) ** 2 * t)

u_true = temperature_torch(X, Tt).flatten()

def rel_l2(model):
    with torch.no_grad():
        u_pred = model(grid).flatten()
    return (torch.linalg.norm(u_pred - u_true) / torch.linalg.norm(u_true)).item()


In [27]:
alpha = 0.1
L = 1.0
T = 1.0
model = Net()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

lambda_res, lambda_boundary, lambda_initial = 1.0, 1.0, 1.0

history = {"total": [], "res": [], "boundary": [], "initial": []}
l2_epochs, l2_values = [], []

    
for epoch in range(5000):
    pts = generate_training_points(1000, L, T)  
    optimizer.zero_grad()
    loss_res, loss_boundary, loss_initial = compute_losses(model, pts, alpha)
    loss = lambda_res * loss_res + lambda_boundary * loss_boundary + lambda_initial * loss_initial
    loss.backward()
    optimizer.step()

    history["total"].append(loss.item())
    history["res"].append(loss_res.item())
    history["boundary"].append(loss_boundary.item())
    history["initial"].append(loss_initial.item())

    if epoch % 50 == 0:
        l2_epochs.append(epoch + 1)
        l2_values.append(rel_l2(model))

    if epoch % 500 == 0:
        print(f"{epoch + 1}: total={loss.item():.2e} res={loss_res.item():.2e} "
              f"boundary={loss_boundary.item():.2e} initial={loss_initial.item():.2e}")

final_l2 = rel_l2(model)
if not l2_epochs or l2_epochs[-1] != epoch + 1:
    l2_epochs.append(epoch + 1)
    l2_values.append(final_l2)
print(f"Final relative L2 error: {final_l2:.4e} ({100 * final_l2:.3f}%)")


1: total=3.74e-01 res=3.99e-03 boundary=3.66e-02 initial=3.33e-01
501: total=1.64e-03 res=1.16e-03 boundary=3.06e-04 initial=1.71e-04
1001: total=2.71e-04 res=2.25e-04 boundary=4.03e-05 initial=5.04e-06
1501: total=1.34e-04 res=9.41e-05 boundary=3.61e-05 initial=3.43e-06
2001: total=8.74e-05 res=5.62e-05 boundary=2.85e-05 initial=2.69e-06
2501: total=8.61e-05 res=5.51e-05 boundary=2.78e-05 initial=3.26e-06
3001: total=7.52e-05 res=5.10e-05 boundary=2.09e-05 initial=3.27e-06
3501: total=6.97e-05 res=4.61e-05 boundary=2.05e-05 initial=3.10e-06
4001: total=6.23e-05 res=4.35e-05 boundary=1.70e-05 initial=1.73e-06
4501: total=6.83e-05 res=4.40e-05 boundary=2.05e-05 initial=3.79e-06
Final relative L2 error: 2.7318e-03 (0.273%)


In [28]:
# Training losses: one value per parameter update.
loss_figure = go.Figure()
for key, values in history.items():
    loss_figure.add_trace(go.Scatter(
        x=list(range(1, len(values) + 1)), y=values,
        mode="lines", name=key,
    ))
loss_figure.update_layout(
    title="PINN training losses",
    xaxis_title="Epoch", yaxis_title="Loss",
    yaxis_type="log", template="plotly_white",
)
loss_figure.show()

# Relative error on the separate evaluation grid.
l2_figure = go.Figure(go.Scatter(
    x=l2_epochs, y=l2_values, mode="lines+markers", name="Relative L2 error",
))
l2_figure.update_layout(
    title="PINN error against the closed-form solution",
    xaxis_title="Epoch", yaxis_title="Relative L2 error",
    yaxis_type="log", template="plotly_white",
)
l2_figure.show()
loss_figure.write_image("pinn_training_losses.png")
l2_figure.write_image("pinn_relative_error.png")

In [29]:
# Compare at the same five positions as Task 3.1.
comparison_times = torch.linspace(0, T, 201).reshape(-1, 1)
comparison_figure = go.Figure()
colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"]
for x0, color in zip(POSITIONS, colors):
    input_pairs = torch.cat([
        torch.full_like(comparison_times, x0), comparison_times,
    ], dim=1)
    with torch.no_grad():
        u_pred = model(input_pairs).flatten().tolist()
    u_exact = temperature_torch(
        torch.full_like(comparison_times, x0), comparison_times,
    ).flatten().tolist()
    for label, values, dash in [("Exact", u_exact, "solid"), ("PINN", u_pred, "dash")]:
        comparison_figure.add_trace(go.Scatter(
            x=comparison_times.flatten().tolist(), y=values,
            mode="lines", name=f"{label}: x={x0:g}", legendgroup=str(x0),
            line={"color": color, "dash": dash},
        ))
comparison_figure.update_layout(
    title="PINN and exact temperature at five rod positions",
    xaxis_title="Time t", yaxis_title="Temperature u(x, t)",
    template="plotly_white",
)
comparison_figure.show()
comparison_figure.write_image("pinn_comparison.png")


## Separate L-BFGS experiment

The Adam experiment above is unchanged. This experiment trains a **new network from scratch**, rather than fine-tuning the Adam model. It uses the same architecture and loss weights. Training points are sampled once and held fixed so repeated closure evaluations use the same objective. A fixed seed makes this experiment reproducible; Adam above did not record its initialization, so this is not a controlled optimizer-only comparison.

Each outer L-BFGS step permits up to 10 internal iterations and may evaluate the closure several times during line search. Consequently, outer steps are not equivalent to Adam iterations. Losses below are measured after each outer step; evaluation error uses the existing independent grid.

Early stopping monitors the fixed-point training loss, not the exact-solution error. Training stops after 15 outer steps without a relative improvement of at least 0.01%, up to a maximum of 500 steps. The best observed training weights are restored for final evaluation and prediction plots. Loss and error histories describe the training trajectory before restoration.


In [ ]:
from time import perf_counter
from copy import deepcopy
import math

with torch.random.fork_rng():
    torch.manual_seed(166)
    model_lbfgs = Net()
    pts_lbfgs = generate_training_points(1000, L, T)

optimizer_lbfgs = torch.optim.LBFGS(
    model_lbfgs.parameters(), lr=1.0, max_iter=10,
    history_size=50, line_search_fn="strong_wolfe",
)
lbfgs_weights = (1.0, 1.0, 1.0)
lbfgs_steps = 500  # Maximum number of outer steps.
early_stop_patience = 15
early_stop_relative_delta = 1e-4  # Require a 0.01% improvement within patience.
progress_reference = float("inf")
best_lbfgs_loss = float("inf")
best_lbfgs_state = None
best_lbfgs_step = 0
stale_lbfgs_steps = 0
history_lbfgs = {key: [] for key in ("total", "res", "boundary", "initial")}
l2_steps_lbfgs = [0]
l2_values_lbfgs = [rel_l2(model_lbfgs)]
lbfgs_counts = {"closure_evaluations": 0}


def closure_lbfgs():
    optimizer_lbfgs.zero_grad(set_to_none=True)
    # Coordinate tensors are leaves too: clear their accumulated gradients.
    pts_lbfgs["x_interior"].grad = None
    pts_lbfgs["t_interior"].grad = None
    components = compute_losses(model_lbfgs, pts_lbfgs, alpha)
    total = sum(weight * component for weight, component in zip(lbfgs_weights, components))
    total.backward()
    lbfgs_counts["closure_evaluations"] += 1
    return total


lbfgs_started = perf_counter()
for step in range(1, lbfgs_steps + 1):
    optimizer_lbfgs.step(closure_lbfgs)
    # step() returns the initial loss, so recompute at the updated weights.
    components = compute_losses(model_lbfgs, pts_lbfgs, alpha)
    component_values = [component.item() for component in components]
    total_value = sum(w * value for w, value in zip(lbfgs_weights, component_values))
    if total_value < best_lbfgs_loss:
        best_lbfgs_loss = total_value
        best_lbfgs_state = deepcopy(model_lbfgs.state_dict())
        best_lbfgs_step = step
    if total_value < progress_reference * (1.0 - early_stop_relative_delta):
        progress_reference = total_value
        stale_lbfgs_steps = 0
    else:
        stale_lbfgs_steps += 1
    history_lbfgs["total"].append(total_value)
    for key, value in zip(("res", "boundary", "initial"), component_values):
        history_lbfgs[key].append(value)
    del components

    if step % 10 == 0 or step == 1 or step == lbfgs_steps:
        l2_steps_lbfgs.append(step)
        l2_values_lbfgs.append(rel_l2(model_lbfgs))
    if step % 50 == 0 or step == 1:
        print(f"L-BFGS step {step}: total={total_value:.3e}, "
              f"relative L2={l2_values_lbfgs[-1]:.3e}")

    if stale_lbfgs_steps >= early_stop_patience:
        print(f"Early stopping at outer step {step}: no significant loss improvement "
              f"for {early_stop_patience} steps.")
        break

# Record the actual stopping step even when it falls between error evaluations.
if l2_steps_lbfgs[-1] != step:
    l2_steps_lbfgs.append(step)
    l2_values_lbfgs.append(rel_l2(model_lbfgs))

lbfgs_seconds = perf_counter() - lbfgs_started
model_lbfgs.load_state_dict(best_lbfgs_state)
print(f"Restored best training loss {best_lbfgs_loss:.3e} from outer step {best_lbfgs_step}.")
final_l2_lbfgs = rel_l2(model_lbfgs)
print(f"Final L-BFGS relative L2: {final_l2_lbfgs:.4e} ({100 * final_l2_lbfgs:.3f}%)")
print(f"Elapsed: {lbfgs_seconds:.1f} s; closure evaluations: {lbfgs_counts['closure_evaluations']}")


L-BFGS step 1: total=3.313e-01, relative L2=5.996e-01
L-BFGS step 50: total=4.876e-06, relative L2=5.503e-04
Early stopping at outer step 75: no significant loss improvement for 15 steps.
Restored best training loss 2.817e-06 from outer step 61.
Final L-BFGS relative L2: 4.7300e-04 (0.047%)
Elapsed: 2.3 s; closure evaluations: 706


In [31]:
lbfgs_loss_figure = go.Figure()
for key, values in history_lbfgs.items():
    lbfgs_loss_figure.add_trace(go.Scatter(
        x=list(range(1, len(values) + 1)), y=values, mode="lines", name=key,
    ))
lbfgs_loss_figure.update_layout(
    title="L-BFGS training losses", xaxis_title="L-BFGS outer step",
    yaxis_title="Loss", yaxis_type="log", template="plotly_white",
)
lbfgs_loss_figure.show()

lbfgs_error_figure = go.Figure(go.Scatter(
    x=l2_steps_lbfgs, y=l2_values_lbfgs, mode="lines+markers", name="Relative L2 error",
))
lbfgs_error_figure.update_layout(
    title="L-BFGS error against the closed-form solution",
    xaxis_title="L-BFGS outer step", yaxis_title="Relative L2 error",
    yaxis_type="log", template="plotly_white",
)
lbfgs_error_figure.show()


In [32]:
lbfgs_comparison_figure = go.Figure()
lbfgs_times = torch.linspace(0, T, 201).reshape(-1, 1)
for x0, color in zip(POSITIONS, colors):
    pairs = torch.cat([torch.full_like(lbfgs_times, x0), lbfgs_times], dim=1)
    with torch.no_grad():
        predicted = model_lbfgs(pairs).flatten().tolist()
        exact = temperature_torch(pairs[:, :1], pairs[:, 1:]).flatten().tolist()
    for label, values, dash in [("Exact", exact, "solid"), ("L-BFGS", predicted, "dash")]:
        lbfgs_comparison_figure.add_trace(go.Scatter(
            x=lbfgs_times.flatten().tolist(), y=values, mode="lines",
            name=f"{label}: x={x0:g}", legendgroup=str(x0),
            line={"color": color, "dash": dash},
        ))
lbfgs_comparison_figure.update_layout(
    title="L-BFGS and exact temperature at five rod positions",
    xaxis_title="Time t", yaxis_title="Temperature u(x, t)", template="plotly_white",
)
lbfgs_comparison_figure.show()

print(f"Adam final relative L2:   {rel_l2(model):.4e}")
print(f"L-BFGS final relative L2: {final_l2_lbfgs:.4e}")
# Distinct filenames preserve the Adam figures.
lbfgs_loss_figure.write_image("pinn_lbfgs_training_losses.png")
lbfgs_error_figure.write_image("pinn_lbfgs_relative_error.png")
lbfgs_comparison_figure.write_image("pinn_lbfgs_comparison.png")


Adam final relative L2:   2.7318e-03
L-BFGS final relative L2: 4.7300e-04
